In [ ]:
from scipy.io.arff import loadarff
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import precision_score
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Conv1D
from tensorflow.keras.layers import MaxPooling1D
from tensorflow.keras.layers import MaxPool2D
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Dense,LSTM
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
print('relevant libraries uploaded')

In [ ]:
fp= '../Dataset/Day1.TCP.arff'
raw_data = loadarff(fp)

In [ ]:
df=pd.DataFrame(raw_data[0])
df

In [ ]:
print("For the Dataset:\n",df['249'].value_counts(),"\n")

In [ ]:
df = df[df['249'] != b'GAMES']

In [ ]:
print("For the Dataset:\n",df['249'].value_counts(),"\n")

In [ ]:
print(list(df.columns))

In [ ]:
df['249'] = df['249'].replace([b'WWW'],1).replace([b'MAIL'],2).replace([b'BULK'],3).replace(
[b'DATABASE'],4).replace([b'P2P'],5).replace([b'SERVICES'],6).replace([b'ATTACK'],7).replace(
[b'MULTIMEDIA'],8).replace([b'INTERACTIVE'],9)

In [ ]:
print("For the Dataset:\n",df['249'].value_counts(),"\n")

In [ ]:
np.any(np.isnan(df))
def clean_dataset(df):
    assert isinstance(df, pd.DataFrame), "df needs to be a pd.DataFrame"
    df.dropna(inplace=True)
    indices_to_keep = ~df.isin([np.nan, np.inf, -np.inf]).any(1)
    return df[indices_to_keep].astype(np.float64)

clean_dataset(df)

In [ ]:
np.any(np.isnan(df))

In [ ]:
X=df.drop(columns=["249"])
y=df["249"]

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
sc=StandardScaler()
X_train=sc.fit_transform(X_train)
X_test=sc.fit_transform(X_test)
print(X.shape)

In [ ]:
X_train=np.array(X_train)
print("Type: ", type(X_train))
print("Type: ", type(y_train))
print("X train shape: ",X_train.shape)
print("X test shape: ",(X_test.shape))
print("Y train shape: ",y_train.shape)
print("Y test shape: ",y_test.shape)

In [ ]:
X_train = X_train.reshape(X_train.shape[0], 12, 1).astype('float32')
X_test = X_test.reshape(X_test.shape[0], 12, 1).astype('float32') 

In [ ]:
model_cnn = Sequential()
model_cnn.add(Conv1D(filters=32, kernel_size=5, input_shape=(12, 1)))
model_cnn.add(MaxPooling1D(pool_size=5 ))
model_cnn.add(Flatten())
model_cnn.add(Dense(1000, activation='relu'))
model_cnn.add(Dense(11, activation='softmax'))

In [ ]:
model_rnn = Sequential()
model_rnn.add(LSTM(128,  activation='relu', input_shape=(12,1),return_sequences=True))
model_rnn.add(Dropout(0.2))
model_rnn.add(LSTM(64, activation='relu'))
model_rnn.add(Dropout(0.2))
model_rnn.add(Dense(32, activation='relu'))
model_rnn.add(Dropout(0.2))
model_rnn.add(Dense(11, activation='softmax'))

In [ ]:
model_dnn = Sequential()
model_dnn.add(Dense(128,input_dim=12,activation="relu"))
model_dnn.add(Dense(64, activation='relu'))
model_dnn.add(Dense(32, activation='relu'))
model_dnn.add(Dense(16, activation='relu'))
model_dnn.add(Dense(11, activation='softmax'))
model_dnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['acc'])

In [ ]:
model_cnn.summary()
model_rnn.summary()
model_dnn.summary()

In [ ]:
model_cnn.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
model_cnn.fit(X_train,y_train,epochs=10)

In [ ]:
model_rnn.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
model_rnn.fit(X_train,y_train,epochs=5)

In [ ]:
model_dnn.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
model_dnn.fit(X_train,y_train,epochs=10)

In [ ]:
predict_cnn=model_cnn.predict(X_test) 
y_pred_cnn=np.argmax(predict_cnn,axis=1)

In [ ]:
predict_rnn=model_rnn.predict(X_test) 
y_pred_rnn=np.argmax(predict_rnn,axis=1)

In [ ]:
predict_dnn=model_rnn.predict(X_test) 
y_pred_dnn=np.argmax(predict_dnn,axis=1)

In [ ]:
cm_cnn = confusion_matrix(y_test, y_pred_cnn)
cm_rnn = confusion_matrix(y_test, y_pred_rnn)
cm_dnn = confusion_matrix(y_test, y_pred_dnn)

In [ ]:
target_names = ["b'WWW'","b'MAIL'","b'BULK'","b'DATABASE'","b'P2P'","b'SERVICES'","b'ATTACK'","b'MULTIMEDIA'","b'INTERACTIVE'"] 
cm_cnn = pd.DataFrame(cm_cnn,index = target_names,columns = target_names)
cm_rnn = pd.DataFrame(cm_rnn,index = target_names,columns = target_names)
cm_dnn = pd.DataFrame(cm_dnn,index = target_names,columns = target_names)

In [ ]:
def createConfusionMatrix(matrix,title):
    plt.figure(figsize=(10,10))
    sns.heatmap(matrix, annot=True, fmt='g')
    plt.title(title, fontsize=20)
    plt.ylabel('Actual Values')
    plt.xlabel('Predicted Values')
    plt.show()

In [ ]:
title = "Brasil CNN Model Confusion Matrix"
createConfusionMatrix(cm_cnn, title)

In [ ]:
title = "Brasil RNN Model Confusion Matrix"
createConfusionMatrix(cm_rnn, title)

In [ ]:
title = "Brasil DNN Model Confusion Matrix"
createConfusionMatrix(cm_dnn, title)

In [ ]:
cnn_report = classification_report(y_test, y_pred_cnn, target_names=target_names)
rnn_report = classification_report(y_test, y_pred_rnn, target_names=target_names)
dnn_report = classification_report(y_test, y_pred_dnn, target_names=target_names)

In [ ]:
print("Brasil Dataset CNN Report\n", cnn_report)

In [ ]:
print("Brasil Dataset RNN Report\n", rnn_report)

In [ ]:
print("Brasil Dataset DNN Report\n", dnn_report)